# JAISP Foundation v7 — Evaluation Notebook

Four checks:
1. **Reconstruction quality** — leave-one-band-out Pearson r, MAE, PSNR, std_ratio across all bands
2. **Cross-instrument reconstruction** — Rubin-only → predict Euclid (and vice versa)
3. **Spatial precision** — r at correct position vs shifted; tested on VIS (0.1"/px) and rubin_r
4. **Band grid plots** — visual truth / prediction / residual for a tile with full band coverage

In [ ]:
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path("../models").resolve()))

from jaisp_dataset_v6 import JAISPDatasetV6
from jaisp_foundation_v7 import ALL_BANDS, EUCLID_BANDS, RUBIN_BANDS, JAISPFoundationV7

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
CHECKPOINT  = "../checkpoints/jaisp_v7_baseline/checkpoint_best.pt"
RUBIN_DIR   = "../data/rubin_tiles_ecdfs"
EUCLID_DIR  = "../data/euclid_tiles_ecdfs"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

N_EVAL_TILES    = None   # None = all tiles
N_SPATIAL_TILES = 20
N_PLOT_TILES    = 2
TOP_FRAC        = 0.10   # bright-pixel fraction from info weights
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def pearson_r(a, b):
    if len(a) < 2:
        return float("nan")
    return float(np.corrcoef(a.ravel(), b.ravel())[0, 1])

def psnr(truth, pred):
    mse = np.mean((truth - pred) ** 2)
    if mse < 1e-12:
        return float("inf")
    return float(10.0 * np.log10(max(np.nanpercentile(np.abs(truth), 99), 1e-6) ** 2 / mse))

def info_mask(info, top_frac=TOP_FRAC):
    return info >= np.nanpercentile(info, (1.0 - top_frac) * 100.0)

def band_pool(sample):
    pool = {}
    pool.update(sample.get("rubin", {}))
    pool.update(sample.get("euclid", {}))
    return pool

def native_res(band):
    if band in RUBIN_BANDS:      return "512×512 @ 0.2\"/px"
    if band == "euclid_VIS":     return "~1050×1050 @ 0.1\"/px"
    return "~350×350 @ 0.3\"/px"

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
device = torch.device(DEVICE)
ckpt   = torch.load(CHECKPOINT, map_location=device)
cfg    = ckpt.get("config", {})

model = JAISPFoundationV7(
    band_names            = cfg.get("band_names", ALL_BANDS),
    stem_ch               = cfg.get("stem_ch", 64),
    hidden_ch             = cfg.get("hidden_ch", 256),
    blocks_per_stage      = cfg.get("blocks_per_stage", 2),
    transformer_depth     = cfg.get("transformer_depth", 4),
    transformer_heads     = cfg.get("transformer_heads", 8),
    fused_pixel_scale_arcsec = cfg.get("fused_pixel_scale_arcsec", 0.8),
).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded checkpoint — epoch {ckpt.get('epoch', '?')}  |  val loss {ckpt.get('best_val_loss', float('nan')):.4f}")

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
dataset     = JAISPDatasetV6(rubin_dir=RUBIN_DIR, euclid_dir=EUCLID_DIR, augment=False, load_euclid=True)
all_indices = list(range(len(dataset)))
print(f"{len(dataset)} tiles loaded")

---
## Check 1 — Leave-one-band-out reconstruction quality

For every tile, leave each available band out and predict it from the rest.
`std_ratio` near 1.0 is healthy; far below 1.0 means the model is predicting a blurred average.
`pred shape` should match native resolution — if VIS shows 512×512 something broke in the decoder.

In [ ]:
@torch.no_grad()
def eval_reconstruction(indices, n_tiles=None):
    if n_tiles:
        indices = indices[:n_tiles]
    per_band = defaultdict(lambda: defaultdict(list))
    for idx in tqdm(indices, desc="Check 1"):
        sample = dataset[idx]
        pool   = band_pool(sample)
        avail  = [b for b in ALL_BANDS if b in pool]
        if len(avail) < 2:
            continue
        for tgt_band in avail:
            ctx = [b for b in avail if b != tgt_band]
            ctx_img = {b: pool[b]["image"].unsqueeze(0).to(device) for b in ctx}
            ctx_rms = {b: pool[b]["rms"].unsqueeze(0).to(device)   for b in ctx}
            tgt_img = pool[tgt_band]["image"].unsqueeze(0).to(device)
            tgt_rms = pool[tgt_band]["rms"].unsqueeze(0).to(device)
            out     = model(ctx_img, ctx_rms, tgt_band, tgt_img, tgt_rms)
            truth   = out["target_norm"].squeeze().cpu().numpy()
            pred    = out["pred"].squeeze().cpu().numpy()
            info    = out["info_weights"].squeeze().cpu().numpy()
            mask    = info_mask(info)
            if mask.sum() < 10:
                continue
            t_b, p_b = truth[mask], pred[mask]
            per_band[tgt_band]["r"].append(pearson_r(t_b, p_b))
            per_band[tgt_band]["mae"].append(float(np.mean(np.abs(p_b - t_b))))
            per_band[tgt_band]["mae_base"].append(float(np.mean(np.abs(t_b))))
            per_band[tgt_band]["psnr"].append(psnr(truth, pred))
            per_band[tgt_band]["std_ratio"].append(float(np.std(p_b) / max(np.std(t_b), 1e-6)))
            per_band[tgt_band]["r_base"].append(pearson_r(t_b, np.zeros_like(t_b)))
            per_band[tgt_band]["pred_h"].append(pred.shape[0])
            per_band[tgt_band]["pred_w"].append(pred.shape[1])
    summary = {}
    for band in ALL_BANDS:
        if band not in per_band:
            continue
        d = per_band[band]
        r = np.asarray(d["r"])
        summary[band] = dict(
            r_mean        = float(np.nanmean(r)),
            r_std         = float(np.nanstd(r)),
            delta_r       = float(np.nanmean(r) - np.nanmean(d["r_base"])),
            mae           = float(np.nanmean(d["mae"])),
            mae_base      = float(np.nanmean(d["mae_base"])),
            psnr          = float(np.nanmean(d["psnr"])),
            std_ratio     = float(np.nanmean(d["std_ratio"])),
            pred_h        = float(np.mean(d["pred_h"])),
            pred_w        = float(np.mean(d["pred_w"])),
            n             = len(r),
        )
    return summary

recon = eval_reconstruction(all_indices, n_tiles=N_EVAL_TILES)

In [ ]:
# Table
HDR = f'{"Band":<14} {"r":>6} {"±":>5} {"Δr":>+6} {"MAE":>6} {"MAE_b":>6} {"PSNR":>6} {"std_r":>6} {"pred shape":>12} {"native":>22} {"N":>4}'
print(HDR)
print("-" * len(HDR))
for band in ALL_BANDS:
    if band not in recon:
        continue
    s = recon[band]
    print(
        f'{band:<14} {s["r_mean"]:>6.3f} {s["r_std"]:>5.3f} {s["delta_r"]:>+6.3f} '
        f'{s["mae"]:>6.2f} {s["mae_base"]:>6.2f} {s["psnr"]:>6.1f} {s["std_ratio"]:>6.3f} '
        f'{s["pred_h"]:.0f}×{s["pred_w"]:.0f}  {native_res(band):>22} {s["n"]:>4}'
    )

In [ ]:
# Bar chart: r per band, coloured by instrument
bands   = [b for b in ALL_BANDS if b in recon]
r_vals  = [recon[b]["r_mean"] for b in bands]
r_errs  = [recon[b]["r_std"]  for b in bands]
colors  = ["steelblue" if b in RUBIN_BANDS else ("gold" if b == "euclid_VIS" else "tomato") for b in bands]
labels  = [b.split("_", 1)[1] for b in bands]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
bars = ax.bar(labels, r_vals, yerr=r_errs, color=colors, capsize=4, edgecolor="white", linewidth=0.5)
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_ylabel("Pearson r (bright pixels)")
ax.set_title("Leave-one-band-out reconstruction r")
ax.set_ylim(bottom=min(0, min(r_vals) - 0.05))
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="steelblue", label="Rubin"),
                   Patch(color="gold",      label="VIS"),
                   Patch(color="tomato",    label="NISP")])

ax2 = axes[1]
std_vals = [recon[b]["std_ratio"] for b in bands]
ax2.bar(labels, std_vals, color=colors, edgecolor="white", linewidth=0.5)
ax2.axhline(1.0, color="gray", lw=1.2, ls="--", label="ideal (1.0)")
ax2.set_ylabel("std_ratio  (pred σ / truth σ)")
ax2.set_title("Prediction spread — near 1.0 is healthy")
ax2.legend()

plt.tight_layout()
plt.show()

---
## Check 2 — Cross-instrument reconstruction

Uses only one instrument as context to predict the other.
- **Rubin → Euclid**: 6 Rubin bands as context, predict each Euclid band
- **Euclid → Rubin**: 4 Euclid bands as context, predict each Rubin band

Lower r than leave-one-out is expected (harder task). Any r > 0 means the model is transferring information across instruments.

In [ ]:
@torch.no_grad()
def eval_cross_instrument(indices, n_tiles=None):
    if n_tiles:
        indices = indices[:n_tiles]
    r2e = defaultdict(lambda: defaultdict(list))   # rubin → euclid
    e2r = defaultdict(lambda: defaultdict(list))   # euclid → rubin

    for idx in tqdm(indices, desc="Check 2"):
        sample = dataset[idx]
        if not sample.get("has_euclid"):
            continue
        pool         = band_pool(sample)
        rubin_avail  = [b for b in RUBIN_BANDS  if b in pool]
        euclid_avail = [b for b in EUCLID_BANDS if b in pool]
        if not rubin_avail or not euclid_avail:
            continue

        for direction, ctx_bands, tgt_bands, store in [
            ("r2e", rubin_avail,  euclid_avail, r2e),
            ("e2r", euclid_avail, rubin_avail,  e2r),
        ]:
            ctx_img = {b: pool[b]["image"].unsqueeze(0).to(device) for b in ctx_bands}
            ctx_rms = {b: pool[b]["rms"].unsqueeze(0).to(device)   for b in ctx_bands}
            for tgt_band in tgt_bands:
                tgt_img = pool[tgt_band]["image"].unsqueeze(0).to(device)
                tgt_rms = pool[tgt_band]["rms"].unsqueeze(0).to(device)
                out     = model(ctx_img, ctx_rms, tgt_band, tgt_img, tgt_rms)
                truth   = out["target_norm"].squeeze().cpu().numpy()
                pred    = out["pred"].squeeze().cpu().numpy()
                mask    = info_mask(out["info_weights"].squeeze().cpu().numpy())
                if mask.sum() < 10:
                    continue
                t_b, p_b = truth[mask], pred[mask]
                store[tgt_band]["r"].append(pearson_r(t_b, p_b))
                store[tgt_band]["std_ratio"].append(float(np.std(p_b) / max(np.std(t_b), 1e-6)))

    def _agg(store):
        return {b: {"r_mean": float(np.nanmean(v["r"])),
                    "std_ratio": float(np.nanmean(v["std_ratio"])),
                    "n": len(v["r"])} for b, v in store.items()}
    return {"rubin_to_euclid": _agg(r2e), "euclid_to_rubin": _agg(e2r)}

cross = eval_cross_instrument(all_indices, n_tiles=N_EVAL_TILES)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (direction, store, tgt_list, color) in zip(axes, [
    ("Rubin → Euclid",  cross["rubin_to_euclid"], EUCLID_BANDS, "gold"),
    ("Euclid → Rubin",  cross["euclid_to_rubin"], RUBIN_BANDS,  "steelblue"),
]):
    bands_here  = [b for b in tgt_list if b in store]
    r_here      = [store[b]["r_mean"] for b in bands_here]
    r_lolo      = [recon[b]["r_mean"] if b in recon else float("nan") for b in bands_here]
    labels_here = [b.split("_", 1)[1] for b in bands_here]
    x = np.arange(len(bands_here))
    ax.bar(x - 0.2, r_lolo,  0.35, label="leave-one-out", color="lightgray", edgecolor="gray")
    ax.bar(x + 0.2, r_here,  0.35, label="cross-instrument", color=color, edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels(labels_here)
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_ylabel("Pearson r")
    ax.set_title(direction)
    ax.legend()

plt.tight_layout()
plt.show()

---
## Check 3 — Spatial precision

Rolls the prediction by N pixels and measures Pearson r against truth. If the model preserves spatial layout, r drops sharply with offset. If r barely changes, the model is predicting a spatially uniform average (bad).

- **VIS** offsets: 2, 5, 10, 20 px → 0.2″, 0.5″, 1″, 2″  
- **Rubin r** offsets: 4, 8, 16, 32 px → 0.8″, 1.6″, 3.2″, 6.4″

In [ ]:
@torch.no_grad()
def eval_spatial_precision(indices, n_tiles=N_SPATIAL_TILES):
    indices = list(indices)[:n_tiles]
    vis_offsets_px   = [2, 5, 10, 20]
    rubin_offsets_px = [4, 8, 16, 32]
    results = {"vis": defaultdict(list), "rubin_r": defaultdict(list)}

    for idx in tqdm(indices, desc="Check 3"):
        sample = dataset[idx]
        pool   = band_pool(sample)
        avail  = [b for b in ALL_BANDS if b in pool]
        if len(avail) < 2:
            continue

        for stream, tgt_band, offsets in [
            ("vis",     "euclid_VIS", vis_offsets_px),
            ("rubin_r", "rubin_r",    rubin_offsets_px),
        ]:
            if tgt_band not in pool:
                continue
            ctx = [b for b in avail if b != tgt_band]
            if not ctx:
                continue
            ctx_img = {b: pool[b]["image"].unsqueeze(0).to(device) for b in ctx}
            ctx_rms = {b: pool[b]["rms"].unsqueeze(0).to(device)   for b in ctx}
            tgt_img = pool[tgt_band]["image"].unsqueeze(0).to(device)
            tgt_rms = pool[tgt_band]["rms"].unsqueeze(0).to(device)
            out     = model(ctx_img, ctx_rms, tgt_band, tgt_img, tgt_rms)
            truth   = out["target_norm"].squeeze().cpu().numpy()
            pred    = out["pred"].squeeze().cpu().numpy()
            mask    = info_mask(out["info_weights"].squeeze().cpu().numpy())
            if mask.sum() < 10:
                continue
            results[stream]["r_correct"].append(pearson_r(truth[mask], pred[mask]))
            _, W = truth.shape
            for dx in offsets:
                p_shifted = np.roll(pred, dx, axis=1)
                crop = slice(0, W - dx)
                t_c, p_c, m_c = truth[:, crop], p_shifted[:, crop], mask[:, crop]
                if m_c.sum() >= 10:
                    results[stream][f"r_{dx}px"].append(pearson_r(t_c[m_c], p_c[m_c]))

    return {s: {k: float(np.nanmean(v)) for k, v in d.items()} for s, d in results.items()}

spatial = eval_spatial_precision(all_indices)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (stream, px_scale, color, title) in zip(axes, [
    ("vis",     0.1, "gold",       "euclid_VIS  (0.1\"/px)"),
    ("rubin_r", 0.2, "steelblue",  "rubin_r     (0.2\"/px)"),
]):
    d = spatial.get(stream, {})
    if not d:
        ax.set_title(f"{title} — no data"); continue
    r_correct = d.get("r_correct", float("nan"))
    offset_keys = sorted([k for k in d if k != "r_correct"],
                         key=lambda k: int(k.split("_")[1].replace("px", "")))
    offsets_arcsec = [int(k.split("_")[1].replace("px", "")) * px_scale for k in offset_keys]
    r_offsets      = [d[k] for k in offset_keys]

    ax.axhline(r_correct, color=color, lw=2, ls="-",  label=f"correct pos (r={r_correct:.3f})")
    ax.plot(offsets_arcsec, r_offsets, "o--", color="gray", lw=1.5, label="shifted")
    ax.fill_between([0] + offsets_arcsec, r_correct, [r_correct] + r_offsets,
                    alpha=0.15, color=color)
    ax.set_xlabel("Shift (arcsec)")
    ax.set_ylabel("Pearson r")
    ax.set_title(f"Spatial precision — {title}")
    ax.legend()
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

---
## Check 4 — Band grid plots

Truth / prediction / residual / info weights for each band on a tile with full coverage.
Output shape is shown in each column title — VIS should read ~1050×1050.

In [ ]:
@torch.no_grad()
def plot_band_grid(tile_idx):
    sample = dataset[tile_idx]
    pool   = band_pool(sample)
    avail  = [b for b in ALL_BANDS if b in pool]
    if len(avail) < 2:
        print(f"Tile {sample['tile_id']}: not enough bands"); return

    n_cols = len(avail)
    fig, axes = plt.subplots(4, n_cols, figsize=(3.0 * n_cols, 11), squeeze=False)
    row_labels = ["Truth (noise units)", "Prediction", "Residual", "Info weights"]

    for col, tgt_band in enumerate(avail):
        ctx = [b for b in avail if b != tgt_band]
        ctx_img = {b: pool[b]["image"].unsqueeze(0).to(device) for b in ctx}
        ctx_rms = {b: pool[b]["rms"].unsqueeze(0).to(device)   for b in ctx}
        out  = model(ctx_img, ctx_rms, tgt_band,
                     pool[tgt_band]["image"].unsqueeze(0).to(device),
                     pool[tgt_band]["rms"].unsqueeze(0).to(device))

        truth = out["target_norm"].squeeze().cpu().numpy()
        pred  = out["pred"].squeeze().cpu().numpy()
        resid = pred - truth
        info  = out["info_weights"].squeeze().cpu().numpy()
        mask  = info_mask(info)

        r = mae = std_r = float("nan")
        if mask.sum() > 10:
            t_b, p_b = truth[mask], pred[mask]
            r     = pearson_r(t_b, p_b)
            mae   = float(np.mean(np.abs(p_b - t_b)))
            std_r = float(np.std(p_b) / max(np.std(t_b), 1e-6))

        lo, hi   = np.nanpercentile(truth, [1, 99])
        lim      = max(float(np.nanpercentile(np.abs(resid), 99)), 1e-3)
        info_hi  = max(float(np.nanpercentile(info, 99.5)), float(info.max()), 1e-6)

        title = f"{tgt_band.split('_',1)[1]}\n{pred.shape[0]}×{pred.shape[1]}"
        if np.isfinite(r):
            title += f"\nr={r:.2f}  mae={mae:.2f}  std={std_r:.2f}"

        for row, (arr, cmap, vr) in enumerate(zip(
            [truth, pred, resid, info],
            ["gray", "gray", "RdBu_r", "inferno"],
            [(lo,hi), (lo,hi), (-lim,lim), (0, info_hi)],
        )):
            ax = axes[row, col]
            ax.imshow(arr, cmap=cmap, vmin=vr[0], vmax=vr[1], origin="lower")
            ax.axis("off")
            if col == 0: ax.set_ylabel(row_labels[row], fontsize=9)
            if row == 0: ax.set_title(title, fontsize=8, pad=3)

    fig.suptitle(f"v7 all-band reconstruction | tile={sample['tile_id']} | {len(avail)} bands",
                 fontsize=11, y=0.995)
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()

# Pick the tiles with the most bands
scored = sorted(
    [(len(band_pool(dataset[i])), i) for i in all_indices if len(band_pool(dataset[i])) >= 2],
    reverse=True,
)
for _, tile_idx in scored[:N_PLOT_TILES]:
    plot_band_grid(tile_idx)

---
## Overall judgement

In [ ]:
rubin_r  = np.nanmean([recon[b]["r_mean"]  for b in RUBIN_BANDS  if b in recon])
euclid_r = np.nanmean([recon[b]["r_mean"]  for b in EUCLID_BANDS if b in recon])
avg_dr   = np.nanmean([recon[b]["delta_r"] for b in recon])
avg_std  = np.nanmean([recon[b]["std_ratio"] for b in recon])

vis_pred_h   = recon.get("euclid_VIS", {}).get("pred_h", 0)
vis_r_cor    = spatial["vis"].get("r_correct", float("nan"))
vis_r_off    = spatial["vis"].get("r_10px",    float("nan"))
vis_gap      = vis_r_cor - vis_r_off

cross_r2e    = np.nanmean([v["r_mean"] for v in cross["rubin_to_euclid"].values()]) if cross["rubin_to_euclid"] else float("nan")

print(f"  Rubin leave-one-out r      : {rubin_r:.3f}  (target >0.75)")
print(f"  Euclid leave-one-out r     : {euclid_r:.3f}  (target >0.65)")
print(f"  Mean Δr vs zero baseline   : {avg_dr:+.3f}  (target >0.30)")
print(f"  Mean std_ratio             : {avg_std:.3f}  (target 0.7–1.1)")
print(f"  VIS pred height            : {vis_pred_h:.0f}    (target ~1050)")
print(f"  VIS spatial gap @ 1\"       : {vis_gap:+.3f}  (target >0.15)")
print(f"  Rubin→Euclid cross-instr r : {cross_r2e:.3f}  (target >0.30)")
print()

ok_recon  = rubin_r > 0.75 and euclid_r > 0.60
ok_res    = vis_pred_h > 900
ok_spatial= vis_gap > 0.15
ok_cross  = cross_r2e > 0.30

if ok_recon and ok_res and ok_spatial and ok_cross:
    verdict = "✓ PASS — ready for downstream astrometry/detection."
elif ok_recon and ok_res:
    verdict = "~ PARTIAL — reconstruction OK; cross-instrument or spatial precision needs work. Train longer."
elif not ok_res:
    verdict = f"✗ FAIL (resolution) — VIS pred_h={vis_pred_h:.0f}, expected ~1050. Check TargetDecoder."
else:
    verdict = "✗ FAIL — model not learning. Check loss curve, LR warmup, data loading."

print(verdict)